# 13 Explainability
Investigator-facing reasons at four levels (transaction, behaviour, beneficiary, graph) from SHAP attributions plus templates; actionable counterfactuals via coherent edits (e.g. "beneficiary older than 90 days"); a deletion-based faithfulness test. Attention weights are never presented as explanations. SHAP describes the model, not the world.

In [1]:
import sys, pathlib, json, warnings; warnings.filterwarnings('ignore')
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
def table(name):
    p = ROOT / 'reports' / 'tables' / name
    return pd.read_csv(p) if p.exists() else None

In [2]:
import json, xgboost as xgb
from presage.data.loaders import load_sim, load_features
from presage.evaluation.metrics import temporal_split, best_threshold
from presage.explainability.explain import Explainer
df, acc = load_sim(); X = load_features(); g = json.load(open(ROOT/'data/processed/sim_v1_feature_groups.json'))
cols = g['txn']+g['behaviour']+g['beneficiary']+g['device']+g['graph']; cust = df.src_is_customer.values==1
tr0,va0,te0 = temporal_split(df.day.values); tr, va, te = tr0&cust, va0&cust, te0&cust; y = df.is_app_fraud.values
m = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=.08, subsample=.8, colsample_bytree=.8, scale_pos_weight=20, tree_method='hist', n_jobs=2, verbosity=0).fit(X.loc[tr, cols].values, y[tr])
thr = best_threshold(y[va], m.predict_proba(X.loc[va, cols].values)[:,1], df.amount.values[va], 'f2')
ex = Explainer(m, cols, g, X.loc[tr & (y==0), cols].sample(3000, random_state=0), thr)
pt = m.predict_proba(X.loc[te, cols].values)[:,1]; i = np.where((pt>=thr)&(y[te]==1))[0][3]
out = ex.explain(X.loc[te, cols].iloc[i], 5); print('risk %.2f' % out['probability']); [print(' -', r['level'], '|', r['text']) for r in out['reasons']]; print(out['counterfactual']['text'])

risk 1.00
 - behaviour | payment time is 9.2 h away from the customer's usual hour
 - behaviour | share of balance moved is 6.3x the customer's previous maximum share
 - beneficiary | beneficiary has paid 12 distinct accounts onward
 - transaction | payment moves 13% of the account balance
 - transaction | 2 failed beneficiary additions in this session
Had amount in line with customer history and small share of balance and beneficiary previously paid by this customer, estimated risk would fall from 1.00 to 0.98.


In [3]:
print(json.dumps(json.load(open(ROOT/'reports/results/explainability_metrics.json')), indent=1) if (ROOT/'reports/results/explainability_metrics.json').exists() else 'run scripts/run_explainability.py')

{
 "surrogate_test_pr_auc": 0.9674069220378435,
 "counterfactual_flip_rate": 0.45318352059925093,
 "mean_edits_when_flipped": 1.3305785123966942,
 "faithfulness": {
  "mean_drop_top_shap": 0.8460100293159485,
  "mean_drop_random": 0.07774924486875534,
  "k": 5,
  "n": 277
 },
 "share_of_flagged_frauds_with_reason_at_level": {
  "transaction": 0.7666666666666667,
  "behaviour": 0.76,
  "beneficiary": 0.4266666666666667,
  "graph": 0.68,
  "device": 0.04
 },
 "n_flagged_frauds": 267
}


In [4]:
p = ROOT/'reports/results/explanations_examples.md'; print(p.read_text()[:3500] if p.exists() else '')

# Example explanations (simulated data)

SHAP attributions describe the model, not causal truth. Counterfactuals are model-based what-ifs.

## Payment 330625 (ceo_fraud) risk 0.99

- [graph] beneficiary was already reported as a mule account (SHAP +7.59)
- [behaviour] payment time is 9.1 h away from the customer's usual hour (SHAP +1.85)
- [behaviour] amount is 5.2x this customer's historical mean payment (SHAP +0.43)
- Counterfactual: Had no confirmed mule accounts near the beneficiary, estimated risk would fall from 0.99 to 0.07.

## Payment 344248 (bank_impersonation) risk 0.96

- [behaviour] payment time is 9.6 h away from the customer's usual hour (SHAP +1.63)
- [behaviour] amount is 22.5x this customer's historical mean payment (SHAP +0.48)
- [transaction] payment moves 13% of the account balance (SHAP +0.45)
- [behaviour] share of balance moved is 1.0x the customer's previous maximum share (SHAP +0.34)
- Counterfactual: Had amount in line with customer history and beneficiary pr

Faithfulness: removing the top-5 SHAP features lowers risk more than removing five random features (see `faithfulness`). That supports, but does not prove, that the explanations track what the model uses.